In [3]:
import requests
import pandas as pd

In [8]:
url = "https://stablecoins.llama.fi/stablecoincharts/all"
response = requests.get(url)
data = response.json()

In [9]:
print(data[0])

{'date': '1511913600', 'totalCirculating': {'peggedUSD': 109970}, 'totalCirculatingUSD': {'peggedUSD': 110105}}


In [17]:
dates = []
supplies = []

for entry in data:
    dates.append(entry['date'])
    supplies.append(entry['totalCirculating']['peggedUSD'])
                

In [18]:
df = pd.DataFrame({
    'date': dates,
    'usd_stablecoin_supply': supplies
})

df['date'] = pd.to_datetime(df['date'].astype(int), unit='s')

print(df.head())

        date  usd_stablecoin_supply
0 2017-11-29               109970.0
1 2017-11-30               109970.0
2 2017-12-01               109970.0
3 2017-12-02               109970.0
4 2017-12-03               109970.0


In [22]:
for entry in data:
    dates.append(entry['date'])
    supplies.append(entry['totalCirculating']['peggedUSD'])

In [24]:
import os
data_path = os.path.expanduser('~/Desktop/Project Git DLT/Project 2/petrodollar-rails/data/raw/stablecoin_supply.csv')
df.to_csv(data_path, index=False)
print("Saved to", data_path)

Saved to /Users/alexhughes/Desktop/Project Git DLT/Project 2/petrodollar-rails/data/raw/stablecoin_supply.csv


In [5]:
import pandas as pd

In [9]:
df = pd.read_csv('/Users/alexhughes/Desktop/Project Git DLT/Project 2/petrodollar-rails/data/raw/stablecoin_supply.csv')
df.head()

,date,usd_stablecoin_supply
0,2017-11-29,109970.0
1,2017-11-30,109970.0
2,2017-12-01,109970.0
3,2017-12-02,109970.0
4,2017-12-03,109970.0


In [11]:
df.tail()

,date,usd_stablecoin_supply
3170,2026-08-04,3.056821e+11
3171,2026-08-05,3.057337e+11
3172,2026-08-06,3.058343e+11
3173,2026-08-07,3.059034e+11
3174,2026-08-08,3.060050e+11


In [12]:
df.shape

(3175, 2)

In [13]:
split_point = int(len(df) * 0.8)
print(f"Train: rows 0 to {split_point}, Test: rows {split_point} to {len(df)}")

Train: rows 0 to 2540, Test: rows 2540 to 3175


In [16]:
train_df = df.iloc[:split_point]
test_df = df.iloc[split_point:]

In [18]:
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

Train shape: (2540, 2), Test shape: (635, 2)


In [20]:
train_supply = train_df['usd_stablecoin_supply'].values
print(f"Train supply shape: {train_supply.shape}, first 5 values: {train_supply[:5]}")

Train supply shape: (2540,), first 5 values: [109970. 109970. 109970. 109970. 109970.]


In [21]:
test_supply = test_df['usd_stablecoin_supply'].values

In [22]:
print(f"Test supply shape: {test_supply.shape}")

Test supply shape: (635,)


In [27]:
from sklearn.preprocessing import MinMaxScaler

# Normalise training data to 0-1 range
scaler = MinMaxScaler()
train_supply_scaled = scaler.fit_transform(train_supply.reshape(-1, 1)).flatten()
test_supply_scaled = scaler.transform(test_supply.reshape(-1, 1)).flatten()

print(f"Scaled train range: {train_supply_scaled.min():.3f} to {train_supply_scaled.max():.3f}")
print(f"Scaled test range: {test_supply_scaled.min():.3f} to {test_supply_scaled.max():.3f}")

Scaled train range: 0.000 to 1.000
Scaled test range: 0.969 to 1.716


In [31]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import numpy as np

# Fit exponential smoothing on training data
model_baseline = ExponentialSmoothing(train_supply_scaled, trend='add', seasonal=None, damped_trend=True)
baseline_fit = model_baseline.fit(optimized=True)

# Forecast test set
baseline_forecast = baseline_fit.forecast(steps=len(test_supply_scaled))

# Calculate error metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(test_supply_scaled, baseline_forecast)
rmse = np.sqrt(mean_squared_error(test_supply_scaled, baseline_forecast))
mape = np.mean(np.abs((test_supply_scaled - baseline_forecast) / test_supply_scaled)) * 100

print(f"Baseline Exponential Smoothing Results:")
print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAPE: {mape:.2f}%")

Baseline Exponential Smoothing Results:
MAE: 0.420223
RMSE: 0.467728
MAPE: 27.08%


In [32]:
def create_windows(data, window_size=30):
    """Create sliding windows of data for time-series forecasting"""
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

window_size = 30
X_train, y_train = create_windows(train_supply_scaled, window_size)
X_test, y_test = create_windows(test_supply_scaled, window_size)

print(f"Training windows: {X_train.shape}")
print(f"Test windows: {X_test.shape}")

Training windows: (2510, 30)
Test windows: (605, 30)
